# Intervention Layer Test

This notebook validates the explicit intervention layer abstraction:
- mask adoption + mask effectiveness
- contact reduction
- multiplicative compilation to network multipliers


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "extensions" / "scenario_api").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate extensions/scenario_api from current working directory")

extensions_dir = repo_root / "extensions"
if str(extensions_dir) not in sys.path:
    sys.path.insert(0, str(extensions_dir))

from scenario_api import (
    MaskProfile,
    MaskAdoptionIntervention,
    ContactReductionIntervention,
    InterventionSet,
    compile_network_multipliers,
)

import pandas as pd
import matplotlib.pyplot as plt

## A) Create mask profile

In [ ]:
mask_profile = MaskProfile(
    name="default",
    effectiveness={
        "surgical": 0.5,
        "ffp2": 0.95,
        "ffp3": 0.99,
    },
)
mask_profile

## B) Create mask adoption intervention

In [ ]:
mask_intervention = MaskAdoptionIntervention(
    name="mask_adoption",
    start=100,
    end=200,
    network_mix={
        "work": {"none": 0.3, "surgical": 0.5, "ffp2": 0.15, "ffp3": 0.05},
        "random": {"none": 0.4, "surgical": 0.4, "ffp2": 0.15, "ffp3": 0.05},
    },
    mask_profile=mask_profile,
)
mask_intervention

## C) Contact reduction intervention

In [ ]:
contact_reduction = ContactReductionIntervention(
    name="contact_reduction",
    start=100,
    end=200,
    multipliers={"work": 0.5, "random": 0.3},
)
contact_reduction

## D) Combine interventions

In [ ]:
intervention_set = InterventionSet([
    mask_intervention,
    contact_reduction,
])
intervention_set

## E) Evaluate at t=120

In [ ]:
multipliers_t120 = compile_network_multipliers(intervention_set, t=120)
multipliers_t120

## F) Plot multipliers over time

In [ ]:
rows = []
for t in range(80, 221):
    m = compile_network_multipliers(intervention_set, t=t)
    rows.append({
        "t": t,
        "work": m.get("work", 1.0),
        "random": m.get("random", 1.0),
    })

df = pd.DataFrame(rows)

display(df.head())
display(df.tail())

# matplotlib 3.2.x + recent pandas works reliably with NumPy arrays.
t_vals = df["t"].to_numpy()
work_vals = df["work"].to_numpy()
random_vals = df["random"].to_numpy()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_vals, work_vals, label="work", linewidth=2)
ax.plot(t_vals, random_vals, label="random", linewidth=2)
ax.axvspan(100, 199, alpha=0.15, label="interventions active")
ax.set_title("Compiled Network Multipliers Over Time")
ax.set_xlabel("t")
ax.set_ylabel("multiplier")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
plt.show()